In [17]:
import pandas as pd
import numpy as np
import scipy.stats as st
import statsmodels.formula.api as smf
import statsmodels.stats as sms
import statsmodels.api as sm

import matplotlib.pyplot as plt
import plotly.express as px
import plotly.io as pio
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, RidgeCV, LassoCV, ElasticNetCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler

In [3]:
base = pio.templates["simple_white"]
custom_template = base.layout.template

custom_template.layout.update(
    margin=dict(l=50, r=0, t=20, b=50),
    xaxis=dict(
        title_standoff=0,
        ticks="outside",
        showgrid=True,
        gridcolor='#bbb'
    ),
    yaxis=dict(
        title_standoff=0,
        ticks="outside",
        showgrid=True,
        gridcolor='#bbb'
    ),
    colorscale=dict(
        sequential=px.colors.sequential.Inferno,
        diverging=px.colors.diverging.curl_r
    ),
    colorway=px.colors.qualitative.T10
)

pio.templates["tight"] = custom_template
pio.templates.default = "tight"

In [4]:
kc = pd.read_csv('kc_house_data.csv')

In [5]:
kc['date'] = pd.to_datetime(kc.date)

sqft_to_sqmeters = 0.09290303997

kc = kc.astype({'sqft_living': float, 'sqft_lot': float, 
                'sqft_above': float, 'sqft_basement': float,
                'sqft_living15': float, 'sqft_lot15': float,
                'yr_renovated': float, 'yr_built': float,
                'bedrooms': float, 'waterfront': float,
                'view': float, 'condition': float, 'grade': float})

kc.loc[:,['sqft_living', 'sqft_lot', 'sqft_above', 'sqft_basement', 'sqft_living15', 'sqft_lot15']] =\
kc.loc[:,['sqft_living', 'sqft_lot', 'sqft_above', 'sqft_basement', 'sqft_living15', 'sqft_lot15']] * sqft_to_sqmeters

kc.columns = ['id', 'date', 'price', 'bedrooms', 'bathrooms', 'sqm_living',
       'sqm_lot', 'floors', 'waterfront', 'view', 'condition', 'grade',
       'sqm_above', 'sqm_basement', 'yr_built', 'yr_renovated', 'zipcode',
       'lat', 'long', 'sqm_living15', 'sqm_lot15']

kc = kc.loc[:, ['id', 'date', 'price', 
            'floors', 'bedrooms', 'bathrooms', 
            'yr_built', 'yr_renovated',
            'waterfront', 'view', 'condition', 'grade', 
            'sqm_living', 'sqm_lot', 
            'sqm_above', 'sqm_basement',
            'sqm_living15', 'sqm_lot15',
            'lat', 'long']]

kc.loc[kc.sqm_basement == 0, 'sqm_basement'] = np.nan
kc.loc[kc.yr_renovated == 0, 'yr_renovated'] = np.nan

In [6]:
kc_log = kc.copy()
kc_log.loc[:, ['price', 'sqm_living', 'sqm_lot', 'sqm_above', 'sqm_basement', 'sqm_living15', 'sqm_lot15']] = \
np.log10(kc_log.loc[:, ['price', 'sqm_living', 'sqm_lot', 'sqm_above', 'sqm_basement', 'sqm_living15', 'sqm_lot15']])

In [7]:
kc_log.yr_renovated = kc_log.yr_renovated.fillna(kc_log.yr_built)
kc_log.loc[:,'sqm_basement'] = kc_log.loc[:,'sqm_basement'].fillna(0)
kc.yr_renovated = kc.yr_renovated.fillna(kc.yr_built)
kc.loc[:,'sqm_basement'] = kc.loc[:,'sqm_basement'].fillna(0)

In [8]:
kc_z=kc_log.copy()
for col in kc_z.drop(columns=['id', 'date']).columns:
    kc_z.loc[:,col] = st.zscore(kc_z.loc[:,col])

In [9]:
# X = kc.drop(columns=['id', 'date', 'price', 'yr_renovated',
#                                 'sqm_above', 'sqm_basement',
#                                 'sqm_living15', 'sqm_lot15',
#                                 'lat', 'long'])
# # X = kc_log.grade.values.reshape(-1, 1)
# y = kc.price
# reg = linear_model.LinearRegression().fit(X, y)
# reg.score(X, y)
# reg.coef_.reshape(-1,1)

In [10]:
# X = kc_log.drop(columns=['id', 'date', 'price', 'lat', 'long'])

X = kc_log.drop(columns=['id', 'date', 'price', 'yr_renovated',
                                'sqm_above', 'sqm_basement',
                                'sqm_living15', 'sqm_lot15',
                                'lat', 'long'])

y = kc_log.price

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

models = {
"Linear Regression": LinearRegression(),
"Ridge": Ridge(alpha=1.0, random_state=42),
"Lasso": Lasso(alpha=0.01, random_state=42, max_iter=10000),
"ElasticNet": ElasticNet(alpha=0.01, random_state=42)
}

def evaluate_model(name, model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_pred_test = model.predict(X_test)
    y_pred_train = model.predict(X_train)

    mse = mean_squared_error(y_test, y_pred_test)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred_test)
    r2 = r2_score(y_test, y_pred_test)

    mse_train = mean_squared_error(y_train, y_pred_train)
    rmse_train = np.sqrt(mse)
    mae_train = mean_absolute_error(y_train, y_pred_train)
    r2_train = r2_score(y_train, y_pred_train)

    results = pd.DataFrame(data = 0.0, index=['r2', 'mse', 'rmse', 'mae'], columns=['test', 'train', 'ratio'])
    results.test=[r2, mse, rmse, mae]
    results.train=[r2_train, mse_train, rmse_train, mae_train]
    results.ratio=[100*r2_train/r2, 100*mse_train/mse, 100*rmse_train/rmse, 100*mae_train/mae]

    print('======== ', name, ' ========\n')
    print(results, '\n')

    print('Coefficients  (abs):')
    print('Intercept    ', round(model.intercept_, 6))
    print(pd.Series(data=np.abs(model.coef_), index=X.columns).sort_values(ascending=False), '\n')

    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(model, X_train_scaled, y_train, scoring='r2', cv=cv)
    print(f"CV R² mean: {cv_scores.mean():.4f} | std: {cv_scores.std():.4f}", '\n')

# Evaluate all models
for name, model in models.items():
    evaluate_model(name, model, X_train_scaled, X_test_scaled, y_train, y_test)

========  Linear Regression  ========

          test     train       ratio
r2    0.658706  0.648666   98.475783
mse   0.018348  0.018252   99.476195
rmse  0.135455  0.135455  100.000000
mae   0.107529  0.107124   99.623273 

Coefficients  (abs):
Intercept     5.665443
grade         0.115678
sqm_living    0.077159
yr_built      0.073898
bathrooms     0.027036
view          0.018897
sqm_lot       0.016440
bedrooms      0.014279
waterfront    0.013274
floors        0.012339
condition     0.009513
dtype: float64 

CV R² mean: 0.6478 | std: 0.0089 

========  Ridge  ========

          test     train       ratio
r2    0.658706  0.648666   98.475762
mse   0.018348  0.018252   99.476237
rmse  0.135455  0.135455  100.000000
mae   0.107530  0.107125   99.623321 

Coefficients  (abs):
Intercept     5.665443
grade         0.115667
sqm_living    0.077153
yr_built      0.073888
bathrooms     0.027038
view          0.018900
sqm_lot       0.016436
bedrooms      0.014273
waterfront    0.013273
floors

In [66]:
np.logspace(start=-4, stop=4, num=25, base=10)

array([1.00000000e-04, 2.15443469e-04, 4.64158883e-04, 1.00000000e-03,
       2.15443469e-03, 4.64158883e-03, 1.00000000e-02, 2.15443469e-02,
       4.64158883e-02, 1.00000000e-01, 2.15443469e-01, 4.64158883e-01,
       1.00000000e+00, 2.15443469e+00, 4.64158883e+00, 1.00000000e+01,
       2.15443469e+01, 4.64158883e+01, 1.00000000e+02, 2.15443469e+02,
       4.64158883e+02, 1.00000000e+03, 2.15443469e+03, 4.64158883e+03,
       1.00000000e+04])

In [47]:
np.logspace(start=5.2, stop=5.3, num=20, base=2)

array([36.75834736, 36.89269197, 37.02752759, 37.16285601, 37.29867902,
       37.43499845, 37.57181609, 37.70913377, 37.84695333, 37.98527659,
       38.12410539, 38.26344158, 38.40328703, 38.54364358, 38.6845131 ,
       38.82589748, 38.96779859, 39.11021832, 39.25315856, 39.39662123])

In [67]:
ridgecv = RidgeCV(alphas=np.logspace(start=-4, stop=4, num=25, base=10), scoring=None, cv=None, store_cv_results=True)

In [68]:
ridgecv.fit(X_train_scaled, y_train)

,alphas,array([1.0000...00000000e+04])
,fit_intercept,True
,scoring,None
,cv,None
,gcv_mode,None
,store_cv_results,True
,alpha_per_target,False


In [ ]:
print(pd.Series(data=np.abs(ridgecv.coef_), index=X.columns).sort_values(ascending=False))

grade         0.115444
sqm_living    0.077015
yr_built      0.073689
bathrooms     0.027076
view          0.018957
sqm_lot       0.016353
bedrooms      0.014165
waterfront    0.013259
floors        0.012367
condition     0.009537
dtype: float64 



In [70]:
ridgecv.alpha_

np.float64(21.54434690031882)

In [71]:
ridgecv.best_score_

np.float64(-0.01828241312415087)

In [74]:
pd.DataFrame(ridgecv.cv_results_)

,0,1,2,3,4,5,6,7,8,9,...,15,16,17,18,19,20,21,22,23,24
0,0.048112,0.048112,0.048112,0.048112,0.048112,0.048112,0.048112,0.048112,0.048112,0.048112,...,0.048071,0.048024,0.047924,0.047710,0.047260,0.046344,0.044588,0.041585,0.037328,0.032637
1,0.002761,0.002761,0.002761,0.002761,0.002761,0.002761,0.002761,0.002761,0.002761,0.002761,...,0.002756,0.002750,0.002738,0.002712,0.002659,0.002556,0.002381,0.002160,0.002091,0.002747
2,0.004644,0.004644,0.004644,0.004644,0.004644,0.004644,0.004644,0.004644,0.004644,0.004644,...,0.004655,0.004668,0.004696,0.004756,0.004885,0.005160,0.005740,0.006941,0.009355,0.014025
3,0.021378,0.021378,0.021378,0.021378,0.021378,0.021378,0.021378,0.021378,0.021378,0.021378,...,0.021388,0.021400,0.021425,0.021480,0.021596,0.021844,0.022356,0.023349,0.025044,0.027433
4,0.000489,0.000489,0.000489,0.000489,0.000489,0.000489,0.000489,0.000489,0.000489,0.000489,...,0.000491,0.000494,0.000499,0.000510,0.000534,0.000583,0.000680,0.000849,0.001075,0.001214
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17285,0.004974,0.004974,0.004974,0.004974,0.004974,0.004974,0.004974,0.004974,0.004974,0.004974,...,0.004972,0.004970,0.004965,0.004954,0.004930,0.004881,0.004776,0.004538,0.003974,0.002795
17286,0.011021,0.011021,0.011021,0.011021,0.011021,0.011021,0.011021,0.011021,0.011021,0.011021,...,0.011020,0.011019,0.011018,0.011015,0.011011,0.011013,0.011046,0.011179,0.011481,0.011831
17287,0.037711,0.037711,0.037711,0.037711,0.037711,0.037711,0.037711,0.037711,0.037711,0.037711,...,0.037667,0.037615,0.037505,0.037271,0.036777,0.035761,0.033780,0.030298,0.025178,0.019316
17288,0.094091,0.094091,0.094091,0.094091,0.094091,0.094091,0.094090,0.094090,0.094090,0.094089,...,0.093899,0.093678,0.093207,0.092205,0.090110,0.085869,0.077797,0.064021,0.044206,0.021849


In [76]:
px.line(pd.DataFrame(ridgecv.cv_results_).apply(np.mean))